# [LAB11] 지도학습 > 시계열 분석 > 주식시세 시계열분석

## 📘 #01. 준비작업

### 📝 [1] 패키지 참조

In [ ]:
!pip install --upgrade yfinance

In [ ]:
from hossam import *
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
import datetime as dt
from pandas import to_datetime, DataFrame, date_range, concat
from prophet import Prophet
from prophet.plot import add_changepoints_to_plot
import holidays
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
# 로딩바
from tqdm.auto import tqdm
# 주식 시세 데이터 수집 모듈
import yfinance as yf

## 📘 #02. 주식 시세 데이터

### 📝 [1] 조회 시작일 데이터 생성

오늘로부터 1년 전 날짜

### 📝 [2] 1년간의 삼성전자 주가 데이터 수집

In [ ]:
today = dt.datetime.now()
start = today - dt.timedelta(days=365*1)
start_day = start.strftime("%Y-%m-%d")
start_day

In [ ]:
origin = yf.download('005930.KS', start=start_day)
print("수입된 데이터 크기:", len(origin))

### 📝 [3] 멀티 컬럼명 제거

오늘 시세를 확인하기 위해 역순 정렬

In [ ]:
origin.tail()

In [ ]:
df = origin.copy()
# 멀티레벨 컬럼을 단일 레벨로 변경 (첫 번째 레벨만 유지)
df.columns = df.columns.get_level_values(0)
df.tail()

## 📘 #03. 1년간의 삼성전자 주식 흐름

### 📝 [1] 1년간 최고가와 최저가 확인 (종가 기준)

### 📝 [2] 최고가를 기록한 날짜와 최저가를 기록한 날짜만 필터링

### 📝 [3] 날짜만 추출

### 📝 [4] 시각화

In [ ]:
max_y = df['Close'].max()
min_y = df['Close'].min()
max_y, min_y

In [ ]:
minmax = df.query("Close == @max_y | Close == @min_y")
minmax

In [ ]:
min_date = minmax.index[0]
max_date = minmax.index[1]
min_date, max_date

In [ ]:
figsize = (1600 / 100, 720 / 100)
fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=my_dpi)
# 주식시세
sb.lineplot(data=df, x=df.index, y="Close")
# 최고점과 최저점 표시
sb.scatterplot(data=minmax, x=minmax.index, y='Close', color='red', s=150, marker='o', ax=ax)
ax.text(min_date, min_y, '[최저점]\n날짜: %s\n종가: %d' % (min_date, min_y), fontsize=12, color='red')
ax.text(max_date, max_y, '[최고점]\n날짜: %s\n종가: %d' % (max_date, max_y), fontsize=12, color='red')
ax.set_title("삼성전자 주가흐름", fontsize=12, pad=8)
ax.set_xlabel("종가", fontsize=8, labelpad=5)
ax.set_ylabel("날짜", fontsize=8, labelpad=5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

## 📘 #03. 데이터 전처리

### 📝 [1] 필요한 변수만 추출

In [ ]:
df1 = df[['Close']].copy()
df1.reset_index(inplace=True)
df1.head()

### 📝 [2] Prophet에 적용할 데이터로 구성

### 📝 [3] 훈련, 검증 데이터 분리

### 📝 [4] 한국 기준 공휴일 데이터

이후 코드는 앞 단원 코드와 동일합니다.

#### ✏ 주말 데이터

In [ ]:
df2 = df1.copy()
df2.rename(columns={"Date": "ds", "Close": "y"}, inplace=True)
df2['ds'] = to_datetime(df2['ds'])
df2.head()

In [ ]:
# 분할 비율
split_ratio = 0.8
# 분할 인덱스
split_idx = int(len(df2) * split_ratio)
# 훈련 / 검증 데이터
train = df2.iloc[:split_idx]
test = df2.iloc[split_idx:]
print("Train 기간:", train['ds'].min(), "~", train['ds'].max())
print("Valid 기간:", test['ds'].min(), "~", test['ds'].max())

#### ✏ 주말 데이터

In [ ]:
start_date = train['ds'].min()
end_date = test['ds'].max()
# 주말 데이터
sat = date_range(start=start_date, end=end_date, freq='W-SAT')
sun = date_range(start=start_date, end=end_date, freq='W-SUN')
weekend = sat.union(sun)
df_weekend = DataFrame({"holiday": "weekend", "ds": weekend.sort_values(), "lower_window": 0, "upper_window": 0})
df_weekend.head()

#### ✏ 공휴일 데이터

In [ ]:
years = list(range(to_datetime(start_date).year, to_datetime(end_date).year + 1))
kr = holidays.KR(years=years)
hd_dict = {"holiday": [], "ds": [], "lower_window": [], "upper_window": []}
for date, name in kr.items():
    hd_dict["holiday"].append(name)
    hd_dict["ds"].append(date)
    hd_dict["lower_window"].append(0)
    hd_dict["upper_window"].append(0)
df_holidays = DataFrame(hd_dict)
df_holidays['ds'] = to_datetime(df_holidays['ds'])
df_holidays.sort_values('ds', inplace=True)
df_holidays.head()

#### ✏ 주말 + 공휴일 데이터 병합

#### ✏ 학습 기간에 포함되는 데이터만 필터링

In [ ]:
holydays_final = concat([df_weekend, df_holidays], ignore_index=True)
holydays_final.sort_values('ds', inplace=True)
holydays_final.reset_index(drop=True, inplace=True)
holydays_final.head(10)

In [ ]:
mask = (holydays_final['ds'] >= start_date) & (holydays_final['ds'] <= end_date)
holyday_final = holydays_final.loc[mask].reset_index(drop=True)
holyday_final.head()

## 📘 #04. Prophet 모델 구현

### 📝 [1] 튜닝하고자 하는 하이퍼파라미터 정의

### 📝 [2] GridSearchCV 구성

In [ ]:
params = ParameterGrid({
    'growth': ['linear'],
    'changepoint_prior_scale': [0.01, 0.1, 1.0],  # <-- 튜닝 대상
    'seasonality_mode': ['additive', 'multiplicative'],  # <-- 튜닝 대상
    'yearly_seasonality': [True],
    'weekly_seasonality': [True],
    'daily_seasonality': [False],
    'holidays': [holyday_final]
})
print('Total Possible Models', len(params))

In [ ]:
%%time
import logging
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
result = []
with tqdm(total=len(params)) as pbar:
    for i, p in enumerate(params):
        pbar.set_description(f"Model {i+1}/{len(params)}")
        m = Prophet(**p)
        m.fit(train)
        future = m.make_future_dataframe(periods=len(test), freq='D')
        forecast = m.predict(future)
        pred = forecast[['ds', 'yhat']][-len(test):]
        score = np.sqrt(mean_squared_error(test['y'].values, pred['yhat'].values))
        result.append({
            "score": score,
            "model": m,
            "params": p,
        })
        pbar.update(1)
# result 배열에서 score가 가장 좋은 모델을 찾는다.
# RMSE가 가장 낮은 모델이 가장 좋은 모델이므로, score가 가장 작은 모델을 찾는다.
best_index = min(result, key=lambda x: x['score'])
best_model = best_index['model']
best_params = best_index['params']
best_score = best_index['score']
print("Best Score (RMSE):", best_score)
print("Best Parameters:", best_params)

### 📝 [3] 예측 데이터 생성

In [ ]:
# 실제 예측 데이터보다 7단계 더 미래까지 예측해보자.(1주일)
future = best_model.make_future_dataframe(periods=len(test)+7, freq='D')
forecast = best_model.predict(future)
forecast.head()

In [ ]:
fig = best_model.plot(forecast, figsize=(20, 7), xlabel='Date', ylabel='Passengers', uncertainty=True)
ax = fig.gca()
add_changepoints_to_plot(ax, best_model, forecast)
ax.set_title("시계열 예측")
# 실제 검증 데이터는 직접 추가한다.
sb.lineplot(data=test, x='ds', y='y', ax=ax, color='#ff6600', linestyle="--", label='test(real)')
plt.show()
plt.close()

In [ ]:
fig = best_model.plot_components(forecast, figsize=(20, 15))
ax = fig.gca()
plt.show()
plt.close()